# Stg Layer — restaurant_mentions (LLM extraction, Incremental)

Extract tên quán + địa chỉ từ **title / description** của mỗi video (`prod.silver.stg_videos`, latest snapshot) bằng `ai_query` (Databricks Foundation Model) → Silver `prod.silver.stg_restaurant_mentions`. Đây là input cho `dim_restaurant` và `fact_video_restaurant_mention` (design 1.3 / 1.4).

**Incremental Flow:**
1. `source_checksum = md5(title || description)` của latest snapshot mỗi video
2. Chỉ gọi LLM cho video **mới** hoặc có title / description **thay đổi** (so với `stg_restaurant_extraction_log`)
3. MERGE kết quả (raw JSON) vào `prod.silver.stg_restaurant_extraction_log` (1 row / video, kể cả video không có quán nào)
4. MERGE (full sync, atomic) từ log → `prod.silver.stg_restaurant_mentions` (1 row / video × tên quán đã normalize); mention của video bị re-extract mà không còn nữa thì bị xóa
5. Verify

**Notes:**
- Không có geocoder: `address` / `ward` / `district` / `city` lấy từ text (LLM copy nguyên văn), `latitude` / `longitude` / `place_id` để NULL ở Gold.
- `mention_source` tính bằng SQL (tên có xuất hiện trong title / description không), không dựa vào LLM.
- `restaurant_name_normalized` = lower, bỏ dấu, 1 space: dùng để dedup.


In [ ]:
%run ../gold/_udfs


In [ ]:
# --- Config ---
VIDEOS_TABLE = "prod.silver.stg_videos"
LOG_TABLE = "prod.silver.stg_restaurant_extraction_log"
MENTIONS_TABLE = "prod.silver.stg_restaurant_mentions"
MODEL = "databricks-qwen3-next-80b-a3b-instruct"  # good Vietnamese, supports responseFormat
MAX_DESCRIPTION_CHARS = 6000

PROMPT = (
    "You extract restaurants (quán ăn / nhà hàng / tiệm) from a Vietnamese YouTube food video. "
    "Only list specific places that the video visits or recommends. "
    "A numbered or listed entry that has its own address counts as a place even when its name is a dish "
    "(e.g. \"BÚN THỊT NƯỚNG MẮM CỦ KIỆU - ĐỊA CHỈ: 1 Nguyễn Trung Trực, Quận 1\"). "
    "Ignore the channel name, sponsors, and dishes that are not tied to a specific place. "
    "Copy addresses as written; city = Hồ Chí Minh when the text says Sài Gòn / SG / HCM. "
    "Use null for any address part the text does not state."
)
RESPONSE_FORMAT = """{"type":"json_schema","json_schema":{"name":"restaurants","schema":{"type":"object","properties":{"restaurants":{"type":"array","items":{"type":"object","properties":{"name":{"type":"string"},"address":{"type":["string","null"]},"ward":{"type":["string","null"]},"district":{"type":["string","null"]},"city":{"type":["string","null"]}},"required":["name","address","ward","district","city"]}}},"required":["restaurants"]},"strict":true}}"""
RESULT_SCHEMA = "STRUCT<restaurants: ARRAY<STRUCT<name: STRING, address: STRING, ward: STRING, district: STRING, city: STRING>>>"

print(f"Videos source: {VIDEOS_TABLE}")
print(f"Extraction log: {LOG_TABLE}")
print(f"Delta target: {MENTIONS_TABLE}")


In [ ]:
# --- Create tables if not exists ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
        video_id         STRING,
        source_checksum  STRING,
        raw_response     STRING,
        restaurant_count INT,
        model_name       STRING,
        extracted_at     TIMESTAMP
    )
    USING DELTA
    COMMENT 'Latest LLM restaurant extraction per video (1 row per video, also when it found no restaurant).'
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {MENTIONS_TABLE} (
        video_id                   STRING,
        restaurant_name            STRING,
        restaurant_name_normalized STRING,
        address                    STRING,
        ward                       STRING,
        district                   STRING,
        city                       STRING,
        mention_source             STRING,
        source_checksum            STRING,
        extracted_at               TIMESTAMP,
        etl_date                   TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
    COMMENT 'Restaurants mentioned per video (grain: video_id x restaurant_name_normalized).'
""")
print("Tables ready ✅")


In [ ]:
# --- Videos to (re-)extract: new, or title / description changed ---
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW videos_to_extract AS
    WITH latest AS (
        SELECT video_id, title, description,
               md5(concat_ws('||', coalesce(title, ''), coalesce(description, ''))) AS source_checksum,
               ROW_NUMBER() OVER (PARTITION BY video_id ORDER BY snapshot_date DESC, updated_at DESC) AS rn
        FROM {VIDEOS_TABLE}
    )
    SELECT l.video_id, l.title, l.description, l.source_checksum
    FROM latest l
    LEFT JOIN {LOG_TABLE} g ON g.video_id = l.video_id
    WHERE l.rn = 1 AND (g.video_id IS NULL OR g.source_checksum <> l.source_checksum)
""")
todo_count = spark.table("videos_to_extract").count()
print(f"Videos to extract this run: {todo_count}")


In [ ]:
# --- Extract with ai_query and MERGE the raw result into the log (LLM is called once per video) ---
if todo_count > 0:
    prompt_sql = PROMPT.replace("'", "''")
    spark.sql(f"""
        MERGE INTO {LOG_TABLE} AS t
        USING (
            SELECT video_id, source_checksum, raw_response,
                   CAST(size(from_json(raw_response, '{RESULT_SCHEMA}').restaurants) AS INT) AS restaurant_count,
                   '{MODEL}' AS model_name, current_timestamp() AS extracted_at
            FROM (
                SELECT video_id, source_checksum,
                       ai_query('{MODEL}',
                                concat('{prompt_sql}', '\\n\\nTITLE: ', coalesce(title, ''),
                                       '\\n\\nDESCRIPTION: ', left(coalesce(description, ''), {MAX_DESCRIPTION_CHARS})),
                                responseFormat => '{RESPONSE_FORMAT}') AS raw_response
                FROM videos_to_extract
            )
        ) AS s
        ON t.video_id = s.video_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✅ Extracted {todo_count} video(s) with {MODEL}")
else:
    print("Nothing new to extract ✅")


In [ ]:
# --- Full-sync MERGE log → mentions (atomic; deletes mentions that a re-extraction dropped) ---
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW mentions_src AS
    WITH exploded AS (
        SELECT g.video_id, g.source_checksum, g.extracted_at, r.*
        FROM {LOG_TABLE} g
        LATERAL VIEW explode(from_json(g.raw_response, '{RESULT_SCHEMA}').restaurants) x AS r
    ),
    cleaned AS (
        SELECT e.video_id, trim(e.name) AS restaurant_name,
               regexp_replace(lower(trim(unaccent_vi(normalize_keyword(e.name)))), '[^a-z0-9 ]', '') AS restaurant_name_normalized,
               nullif(trim(e.address), '') AS address, nullif(trim(e.ward), '') AS ward,
               nullif(trim(e.district), '') AS district, nullif(trim(e.city), '') AS city,
               e.source_checksum, e.extracted_at,
               lower(unaccent_vi(coalesce(v.title, ''))) AS title_u,
               lower(unaccent_vi(coalesce(v.description, ''))) AS description_u
        FROM exploded e
        LEFT JOIN (
            SELECT video_id, title, description,
                   ROW_NUMBER() OVER (PARTITION BY video_id ORDER BY snapshot_date DESC, updated_at DESC) AS rn
            FROM {VIDEOS_TABLE}
        ) v ON v.video_id = e.video_id AND v.rn = 1
        WHERE length(trim(e.name)) > 1
    ),
    tagged AS (
        SELECT *,
            CASE
                WHEN instr(title_u, lower(unaccent_vi(restaurant_name))) > 0
                 AND instr(description_u, lower(unaccent_vi(restaurant_name))) > 0 THEN 'both'
                WHEN instr(title_u, lower(unaccent_vi(restaurant_name))) > 0 THEN 'title'
                ELSE 'description'
            END AS mention_source,
            ROW_NUMBER() OVER (PARTITION BY video_id, restaurant_name_normalized ORDER BY address IS NULL, restaurant_name) AS rn
        FROM cleaned
        WHERE restaurant_name_normalized <> ''
    )
    SELECT video_id, restaurant_name, restaurant_name_normalized, address, ward, district, city,
           mention_source, source_checksum, extracted_at, current_timestamp() AS etl_date
    FROM tagged WHERE rn = 1
""")

spark.sql(f"""
    MERGE INTO {MENTIONS_TABLE} AS t
    USING mentions_src AS s
    ON t.video_id = s.video_id AND t.restaurant_name_normalized = s.restaurant_name_normalized
    WHEN MATCHED AND t.source_checksum <> s.source_checksum THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
print(f"✅ Merged into {MENTIONS_TABLE}")


In [ ]:
# --- Verify ---
stats = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT video_id, restaurant_name_normalized) AS n_grain,
           COUNT(DISTINCT video_id) AS videos, COUNT(address) AS with_address
    FROM {MENTIONS_TABLE}
""").collect()[0]
log = spark.sql(f"SELECT COUNT(*) AS n, COUNT_IF(restaurant_count = 0) AS empty FROM {LOG_TABLE}").collect()[0]
print(f"📊 Extraction log: {log['n']} videos ({log['empty']} without any restaurant)")
print(f"📊 {MENTIONS_TABLE}: {stats['n']} mentions in {stats['videos']} videos, {stats['with_address']} with an address")
print(f"📊 Grain check (video_id × restaurant_name_normalized): {stats['n']} rows vs {stats['n_grain']} distinct "
      + ("✅" if stats["n"] == stats["n_grain"] else "❌ duplicates"))
print("\n--- stg_restaurant_mentions ETL complete ---")
